# AutoVSF Workstation Platform v2.0 Launcher
Run the cells below to mount Google Drive, bootstrap the Ubuntu Workstation environment in background, and launch the 2 Workstation Portals.

In [ ]:
# Step 1: Connect to Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
WORKDIR = '/content/drive/MyDrive/AutoVSF'
os.makedirs(WORKDIR, exist_ok=True)
%cd $WORKDIR

if not os.path.exists('autovsf-colab-gui'):
    !git clone https://github.com/lionc2240/autovsf-colab-gui.git
%cd autovsf-colab-gui
!git pull

In [ ]:
# Step 2: Bootstrap Workstation Environment in Background (Non-blocking)
import subprocess, time, os, shutil
from google.colab.output import eval_js

!chmod +x bootstrap/setup_environment.sh
subprocess.Popen(['bash', 'bootstrap/setup_environment.sh'], stdout=open('/tmp/setup_env.log', 'w'), stderr=subprocess.STDOUT)

# Launch ttyd log viewer as soon as ttyd is installed in background
log_launcher = "while ! command -v ttyd >/dev/null 2>&1; do sleep 1; done; pkill ttyd || true; ttyd -p 7681 tail -f /tmp/setup_env.log"
subprocess.Popen(['bash', '-c', log_launcher], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(1)
url_log = eval_js('google.colab.kernel.proxyPort(7681)')
print('=' * 75)
print(f'LINK CHECK SETUP PROGRESS (Live Log Terminal): {url_log}')
print('=' * 75)
print('[SUCCESS] Setup running in background. You can proceed to Step 3!')

In [ ]:
# Step 3: Launch Workstation & Output 2 Access Portals (Terminal & Desktop GUI)
import subprocess, time, os, shutil
from google.colab.output import eval_js

# Wait until background setup finishes installing desktop dependencies
while not (shutil.which('xfce4-session') or shutil.which('openbox') or shutil.which('openbox-session')):
    print('[WAIT] Background setup is installing packages... (re-checking in 4s)')
    time.sleep(4)

# Start Virtual Framebuffer Xvfb
!pkill Xvfb || true
subprocess.Popen(['Xvfb', ':1', '-screen', '0', '1280x800x24'])
os.environ['DISPLAY'] = ':1'
time.sleep(1)

# Start Desktop Session (XFCE / Openbox fallback)
desktop_cmd = 'xfce4-session' if shutil.which('xfce4-session') else 'openbox-session'
subprocess.Popen([desktop_cmd], env=os.environ)
time.sleep(1)

# Start x11vnc & noVNC Web Server (Port 6080)
subprocess.Popen(['x11vnc', '-display', ':1', '-forever', '-shared', '-nopw', '-rfbport', '5900'])
subprocess.Popen(['websockify', '--web=/usr/share/novnc/', '6080', 'localhost:5900'])

# Start ttyd Web Terminal Server (Port 7681)
!pkill ttyd || true
if shutil.which('ttyd'):
    subprocess.Popen(['ttyd', '-p', '7681', 'bash'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# Launch AutoVSF Desktop GUI App
subprocess.Popen(['python3', '-m', 'autovsf_gui.app'], env=os.environ)
time.sleep(1)

url_ttyd = eval_js('google.colab.kernel.proxyPort(7681)')
url_novnc = eval_js('google.colab.kernel.proxyPort(6080)')

print('=' * 75)
print(f'LINK 1 [TERMINAL WEBUI (ttyd)]:  {url_ttyd}')
print(f'LINK 2 [DESKTOP GUI (noVNC)]:   {url_novnc}/vnc.html')
print('=' * 75)

In [ ]:
# @title Step 4: Quick YouTube Download & Enqueue AutoVSF Job
youtube_url = "https://www.youtube.com/watch?v=..." #@param {type:"string"}
enable_translation = True #@param {type:"boolean"}

if youtube_url and ("youtube.com" in youtube_url or "youtu.be" in youtube_url):
    from autovsf_core.engine.youtube.downloader import YouTubeDownloader
    from autovsf_core.domain.crop import CropProfile
    from autovsf_core.domain.job import Job
    from autovsf_core.queue.manager import queue_manager

    print(f'[YOUTUBE] Downloading video (720p+, original title)...')
    dl = YouTubeDownloader()
    video_file = dl.download_video(youtube_url, min_height=720)
    print(f'[SUCCESS] Downloaded: {video_file}')

    job = Job(video_path=video_file, crop_profile=CropProfile(top=0.2, bottom=0.0), enable_translation=enable_translation)
    job_id = queue_manager.enqueue_job(job)
    print(f'[QUEUE] Enqueued Job ID: {job_id}')
else:
    print('[INFO] Please enter a valid YouTube URL above.')

In [ ]:
# @title Step 5: Paste from Clipboard (Ultimate Auto-Sync) {run: "auto"}
import ipywidgets as widgets
from IPython.display import display
import os, shutil

# Ensure xclip is installed
if not shutil.which('xclip'):
    os.system('apt-get install -y xclip > /dev/null 2>&1')

text_input = widgets.Textarea(
    value='',
    placeholder='Paste text from host computer here (Ctrl+V)...',
    description='Content:',
    disabled=False,
    layout=widgets.Layout(width='100%', height='120px')
)

output = widgets.Output()

def on_value_change(change):
    with output:
        output.clear_output()
        text = change['new']
        if not text.strip():
            return

        with open('/tmp/clipboard.txt', 'w', encoding='utf-8') as f:
            f.write(text)

        os.system('DISPLAY=:1 xclip -selection clipboard < /tmp/clipboard.txt')

        print('[SUCCESS] Synced content directly into noVNC Desktop Clipboard!')
        print('[INFO] Switch to noVNC tab and press Ctrl+Shift+V or Right Click -> Paste.')

text_input.observe(on_value_change, names='value')
display(text_input, output)